# 🎤 Voice Conversion on Kaggle: Seed-VC and RVC

Same pipeline as the Colab notebook: the vocal is split off, converted to another voice, and laid back over the untouched instrumental.

📖 **Full step-by-step guide:** [`voice-conversion/GUIDE.md`](https://github.com/guitorte/audio/blob/ccr-8be874fb-rdbsnf/voice-conversion/GUIDE.md) · 🇧🇷 [em português](https://github.com/guitorte/audio/blob/ccr-8be874fb-rdbsnf/voice-conversion/GUIDE.pt-BR.md), covering setup, every setting, training and troubleshooting.

### One-time setup
1. **Upload your files as a Kaggle Dataset** (*Create ▸ New Dataset*). Put everything **at the root**, no folders needed:
   ```
   my-songs/
   ├── relaxa.mp3          ← songs to convert
   ├── displicente.wav     ← a voice clip for Seed-VC (pick it by number or name in cell 2)
   ├── maria.pth           ← RVC model you already have (+ maria.index)
   ├── maria take 1.wav    ← recordings to train an RVC voice
   └── maria take 2.wav       (pick them by name, e.g. "maria take", or numbers in cell 4)
   ```
   Every audio file shows up numbered in cell 1's list; each cell takes those numbers or parts of names.
2. In this notebook: **Add Input** ▸ your dataset.
3. **Settings** (right panel): *Accelerator* ▸ **GPU T4 x2** (not P100), and *Internet* ▸ **On** (needs a phone-verified account).

### Using it
- Run cell 1, then set `RUN = True` in the cell you want (2, 3 or 4) and run it. The `RUN` switches keep a full *Save & Run All* from launching every task at once.
- Results go to **`/kaggle/working/`**: `converted/` and `voices/` (trained models). Kaggle keeps them **only if you save a version**: *Save Version ▸ Save & Run All*, or download them (cell 6 zips them).
- **Long RVC training:** run it as *Save & Run All* (up to ~12 h, keeps running with the tab closed). To continue later, **Add Input ▸ your notebook's output** and run cell 4 again with the same settings: it resumes from the saved checkpoints.

⚖️ Only convert voices you have the right to use. Check Kaggle's terms for what its free GPUs may be used for.

Branch: `ccr-8be874fb-rdbsnf`

In [ ]:
# 1 · Setup: find your dataset, list songs and voices
INPUT = ""   # empty = auto-detect; or the dataset folder, e.g. "my-songs"

import os, subprocess, sys
REPO, BRANCH = "/tmp/audio", "ccr-8be874fb-rdbsnf"   # outside /kaggle/working so it isn't saved as output
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                       "https://github.com/guitorte/audio.git", REPO]).returncode:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/guitorte/audio.git", REPO], check=True)
sys.path.insert(0, f"{REPO}/voice-conversion")
import vc
U = vc.U

print("GPU:", vc.kaggle_setup() or "none")
FOLDER = vc.kaggle_input(INPUT)
OUT = vc.KAGGLE_OUTPUT
FILES = U.list_audio(FOLDER)
print(f"\n📁 {FOLDER}  ({len(FILES)} songs)  →  results in {OUT}")
for i, f in enumerate(FILES, 1):
    print(f"{i:3d}. {f.name}")
print()
print(vc.describe(vc.scan_voices(FOLDER, OUT)))
RESULTS = []

In [ ]:
# 2 · Seed-VC: convert with a reference clip (no training)
RUN = False
SONGS = "1"              # "all", numbers ("1, 3-4") or parts of names
REFERENCE = ""           # number or part of the name of the voice clip, from cell 1's list
SEMITONES = 0            # -12..12; +12 = male song for a female voice
DIFFUSION_STEPS = 30     # 30-50 recommended for singing
AUTO_OCTAVE = False      # optional: add a whole-octave shift if the two vocals sit far apart
SINGING_MODEL = True     # False = spoken word (22 kHz speech model)
SEPARATE_VOCALS = True   # False if the songs are already a cappella
ISOLATE_REFERENCE_VOCAL = True
VOCAL_GAIN_DB = 0.0
OVERWRITE = False

if RUN:
    opts = vc.Options(separate=SEPARATE_VOCALS, vocal_gain_db=VOCAL_GAIN_DB, overwrite=OVERWRITE,
                      singing=SINGING_MODEL, semitones=SEMITONES, steps=DIFFUSION_STEPS,
                      isolate_reference=ISOLATE_REFERENCE_VOCAL, auto_octave=AUTO_OCTAVE)
    RESULTS = vc.convert(U.select_files(FILES, SONGS), FOLDER, "seedvc", REFERENCE, opts, out=OUT)
    for r in RESULTS:
        print(f"\n{os.path.basename(r.source)} → {r.output}", *[f"   · {n}" for n in r.notes + [r.skipped] if n], sep="\n")

In [ ]:
# 3 · RVC: convert with a voice model
RUN = False
SONGS = "1"
MODEL = ""               # name of a .pth (in the dataset or trained here), without extension
PITCH = 0                # -12..12
INDEX_RATE = 0.5         # 0..1, needs the .index; higher = closer to the voice, sometimes less clear
PROTECT = 0.33           # 0..0.5; lower = stronger protection of breaths/consonants
AUTO_OCTAVE = False      # optional octave check, added to PITCH...
PITCH_REFERENCE = ""     # ...against this clip of the model's voice (number or name from cell 1)
SEPARATE_VOCALS = True
VOCAL_GAIN_DB = 0.0
OVERWRITE = False

if RUN:
    opts = vc.Options(separate=SEPARATE_VOCALS, vocal_gain_db=VOCAL_GAIN_DB, overwrite=OVERWRITE,
                      semitones=PITCH, index_rate=INDEX_RATE, protect=PROTECT,
                      auto_octave=AUTO_OCTAVE, pitch_reference=PITCH_REFERENCE)
    RESULTS = vc.convert(U.select_files(FILES, SONGS), FOLDER, "rvc", MODEL, opts, out=OUT)
    for r in RESULTS:
        print(f"\n{os.path.basename(r.source)} → {r.output}", *[f"   · {n}" for n in r.notes + [r.skipped] if n], sep="\n")

In [ ]:
# 4 · RVC: train a voice model from your recordings
RUN = False
DATASET = ""             # recordings from cell 1's list: part of their names ("maria take") or numbers ("4-9")
MODEL_NAME = ""          # name for the new voice; required when DATASET is numbers (10+ min of audio works best)
ISOLATE_VOCALS = True    # False for clean a cappella takes
EPOCHS = 200
SAVE_EVERY = 10
BATCH_SIZE = 8
SAMPLE_RATE = 40000      # 32000, 40000 or 48000

# Checkpoints are mirrored to /kaggle/working/voices/.training/. To resume in a later session,
# Add Input ▸ this notebook's saved output, then run this cell again with the same settings.
if RUN:
    topts = vc.TrainOptions(isolate_vocals=ISOLATE_VOCALS, epochs=EPOCHS, save_every=SAVE_EVERY,
                            batch_size=BATCH_SIZE, sample_rate=SAMPLE_RATE)
    vc.train(FOLDER, DATASET, MODEL_NAME, topts, out=OUT)
    print()
    print(vc.describe(vc.scan_voices(FOLDER, OUT)))

In [ ]:
# 5 · Listen: original vs converted (last run of cell 2 or 3)
START_AT_SECONDS = 45
CLIP_SECONDS = 30

import numpy as np
from IPython.display import Audio, Markdown, display

def excerpt(path):
    return np.clip(U.excerpt(path, START_AT_SECONDS, CLIP_SECONDS, 44100).T, -1, 1)

for r in RESULTS:
    if not r.output:
        continue
    display(Markdown(f"### {os.path.basename(r.output)}"))
    for title, path in (("Original", r.source), ("Converted", r.output), ("Converted vocal only", r.vocals)):
        if path:
            display(Markdown(f"**{title}**")); display(Audio(excerpt(path), rate=44100, normalize=False))

In [ ]:
# 6 · Zip the results for download (Output panel ▸ results.zip)
import shutil
zip_path = shutil.make_archive(str(OUT / "results"), "zip", OUT, "converted") if (OUT / "converted").is_dir() else None
print(zip_path or "Nothing converted yet.")

In [ ]:
# 7 · Optional: octave check only (no conversion)
RUN = False
SOURCE = ""              # the vocal you'll convert (number or name from cell 1)
TARGET = ""              # a clip of the target voice
ISOLATE = False          # True only for full songs instead of clean vocal stems

if RUN:
    vc.pitch_check(FOLDER, SOURCE, TARGET, out=OUT, isolate=ISOLATE)

### Tips
- **Pitch range matters most.** If the target voice is much higher or lower than the original singer, use `SEMITONES` / `PITCH` (±12 = one octave).
- **Weekly GPU quota** (about 30 h): Seed-VC conversion takes minutes. RVC training is what uses the hours.
- **Dataset updates:** after adding files to your dataset, create a new dataset version and re-attach it (or restart the session) so the notebook sees them.